In [1]:
# Cell 1: Imports and Setup
import os
import json
import pprint
from collections import defaultdict
from laser_shock_lab import LaserShockLab

# Set a pretty printer for later
pp = pprint.PrettyPrinter(indent=2)

ModuleNotFoundError: No module named 'laser_shock_lab'

In [ ]:
# Cell 2: Initialize the LaserShockLab model and fetch FileMaker records
lab = LaserShockLab()

# Create all GEMD objects from FileMaker
lab.create_gemd_objects()

In [ ]:
# Cell 3: Extract raw record data from layouts
layouts = [
    "Glass ID", "Foil ID", "Epoxy ID", "Spacer ID",
    "Flyer Cutting Program", "Spacer Cutting Program", "Flyer Stack",
    "Sample", "Launch Package", "Experiment"
]

layout_to_records = {}
for layout in layouts:
    print(f"Fetching records from layout: {layout}")
    records = lab._LaserShockLab__get_filemaker_records(layout_name=layout, n_max_records=1000)
    layout_to_records[layout] = records

print("Finished fetching records.")

In [ ]:
# Cell 4: Derive schema per layout
layout_schema = {}

def infer_field_types(records):
    types = defaultdict(set)
    for record in records:
        for key, value in record.items():
            types[key].add(type(value).__name__)
    return {k: list(v) for k, v in types.items()}

for layout, records in layout_to_records.items():
    print(f"\nSchema for layout: {layout}")
    inferred_types = infer_field_types(records)
    layout_schema[layout] = inferred_types
    pp.pprint(inferred_types)

In [ ]:
# Cell 5: Propose a MongoDB document structure per layout
layout_mongo_schemas = {}

for layout, schema in layout_schema.items():
    mongo_doc = {
        "type": "object",
        "description": f"Document schema for {layout} layout",
        "properties": {}
    }
    for field, types in schema.items():
        mongo_type = "string"
        if "int" in types:
            mongo_type = "integer"
        elif "float" in types:
            mongo_type = "number"
        elif "bool" in types:
            mongo_type = "boolean"
        mongo_doc["properties"][field] = {"type": mongo_type}
    layout_mongo_schemas[layout] = mongo_doc

# Print one example
layout_example = "Sample"
print(f"\nMongoDB Schema for layout: {layout_example}")
pp.pprint(layout_mongo_schemas[layout_example])

In [ ]:
# Cell 6: Suggest Girder integration steps
from IPython.display import Markdown

md = """
### ✅ Girder Integration Recommendations

To integrate this metadata into a Girder collection or item:

1. **Set metadata on an item** using the Girder REST API:

```bash
curl -X PUT "http://<girder_host>/api/v1/item/<item_id>/metadata" \\
     -H "Girder-Token: <your_api_key>" \\
     -H "Content-Type: application/json" \\
     -d '<metadata_json>'
```

2. **Example Python snippet** using `requests`:

```python
import requests

item_id = "<your_item_id>"
metadata = layout_to_records["Sample"][0]  # Example: first record
headers = {
    "Girder-Token": "<your_token>"
}
url = f"http://<girder_host>/api/v1/item/{item_id}/metadata"
resp = requests.put(url, headers=headers, json=metadata)
print(resp.status_code, resp.json())
```

3. **Repeat** for other records/layouts, optionally attaching metadata to folders, items, or users.

> Consider validating metadata documents against the proposed JSON schema before inserting.
"""

display(Markdown(md))